In [ ]:
size                 = None
file_path_MK         = None
only_inside          = None
keep_engulfing_voids = None
file_path_ellips_Z   = None

In [ ]:
%run ./Analysis___Ellipsoids_and_Fits_1___Functions.ipynb
%run ./0___Reference___Plots.ipynb
%run ./0___Reference___Shapes.ipynb

---
---
---
---
---
---
---
---
---
---

In [ ]:
with open(file_path_MK+"fg___100.pk",  'rb') as f: fg_100 = pkl.load(f)
# Re-order the void indices so they start from 0 and no gaps.
fg_100[fg_100 != -2] = np.searchsorted(np.sort(np.unique(fg_100[fg_100 != -2])), fg_100[fg_100 != -2].ravel())
no_voids = np.max(fg_100)+1   # +1 for the 0 index

---

In [ ]:
voids_used             = np.ones( no_voids, dtype=bool)
BAD_voids              = np.zeros(no_voids, dtype=bool)
voids_multiple_patches = np.zeros(no_voids, dtype=bool)
l_center = []; l_radii = []; l_evecs = []; l_v = []; l_i1 = []

# For each void...
for i1 in range(no_voids):

    
    ### Find edges of all walls' patches and remove engulfed patches
    ################################################################
    
    # We find all its coordinates that neighbour either a wall or the edge of the grid.
    voids_edge_coords = voids_edge_coords_fct(fg_100, i1, size)

    # We use label() to partition the later into patches.
    fg_edges = np.zeros((size,size,size), dtype=bool)
    for i,j,k in voids_edge_coords: fg_edges[i][j][k] = True
    labeled_array, no_patches = label(fg_edges, structure=generate_binary_structure(3,3))

    # To the very best of my knowledge and research, label() is deterministic and scans the grid in the same pattern in increasing index order.
    # Thus, the two labeled_array's must match between the full patches and just their edges.
    labeled_array1, no_patches1 = label(fg_100 == i1, structure=generate_binary_structure(3,3))

    # Now, if we have multiple such edge patches, it may be that some of them are completely engulfed within others.
    # (I personally did a sanity check and all the walls are defined correctly as touching two different voids and there are no other anomalies. So...)
    # The only reason this happens is because there is a different void completely engulfed inside this one. It may be inside a single patch or between multiple at
    #    or through the edges, but this is the only reason this happens.
    # You can see if you run label() with fg_edges filled with all the void coordinates, not just its edges, that the number of patches decreases in such cases.
    # We check for this.
    if no_patches >= 2:
        
        # If we keep it, we must find the inner patches and remove them.
        if not only_inside:
            # There are two cases:
            #    1. The inner patch is engulfed inside a single patch:
            #          min(coords_inner) >  min(coords_outter) and max(coords_inner) <  max(coords_outter).
            #    2. The inner patch sits at the edge and is thus engulfed by multiple (outer) patches:
            #          min(coords_inner) >= min(coords_outter) and max(coords_inner) <= max(coords_outter), where equality happens only at 0 and size-1.
            #       This case is more complicated, topologically. However, we have already found by now through the bounds comparing each pair of
            #          patches which one must be the engulfed one. Thus, we check in labeled_array1 if they share the same index and if so, engulfed it is! 

            # Find the bounds for all patches.
            bounds = get_patch_bounds(labeled_array, no_patches)

            # See if there are engulfed patches and if so which ones.
            engulfed_patches = find_engulfed_patches(bounds, labeled_array, no_patches, labeled_array1, size)

            if np.sum(engulfed_patches) > 0:
                if keep_engulfing_voids:
                    # Remove them.
                    voids_edge_coords_fin = []
                    for i,j,k in voids_edge_coords:
                        if not engulfed_patches[labeled_array[i][j][k]-1]: voids_edge_coords_fin.append([i,j,k])
                    voids_edge_coords = voids_edge_coords_fin
        
                    i2_del = 0
                    for i2 in range(no_patches):
                        if engulfed_patches[i2]: labeled_array[labeled_array == i2+1] = 0;   i2_del += 1
                        else:                    labeled_array[labeled_array == i2+1] = i2+1-i2_del
                    
                    no_patches -= np.sum(engulfed_patches)
                
                else: voids_used[i1] = False; continue
        else:         voids_used[i1] = False; continue

    with open(file_path_ellips_Z+"voids_edge_coords/"+str(i1)+".pk", 'wb') as f: pkl.dump(voids_edge_coords, f)
    print(file_path_ellips_Z+"voids_edge_coords/"+str(i1)+".pk")
    
    



    ### Check for infinite loops inside each void
    #############################################

    labeled_array  -= 1 # so we can match each void index with its value
    labeled_array1 -= 1
    
    # If we still have multiple patches (they weren't just from engoulfed voids), we check for infinite loops
    if no_patches >= 2:
        voids_multiple_patches[i1] = True
        
        
        # For each part, we find its connections (through edges, ofc) with other patches and the directions in which that they occur.
        # There are two ways infinite loops can form (BAD_void) at this stage:
        #    1. If a patch connects to itself, it must do so on opposite sides (ofc) so it forms an infinite loop.
        #    2. If a patch connects to another on two different sides, it automatically creates an infinite loop.
        # We check these conditions inside check_this_patch(), which returns the the result as the BAD_void.
        BAD_void = False
        all_connections = []; all_directions = []
        for p_i in range(no_patches):
            connections, directions, BAD_void = check_this_patch(p_i, labeled_array, no_patches, size)
            if BAD_void: break
            all_connections.append(connections); all_directions.append(directions)
        if BAD_void: continue

        with open(file_path_ellips_Z+"all_connections/"+str(i1)+".pk", 'wb') as f: pkl.dump(all_connections, f)
        with open(file_path_ellips_Z+"all_directions/" +str(i1)+".pk", 'wb') as f: pkl.dump(all_directions,  f)



        ### A few comments
        # 1. Make sure we do not pick instant loops: (0->1->0)
        # 2. Every odd loop is infinite. (We don't implement this, as it would introduce complexity with no beneift, but worth noting. :D))

            
        # We start with index 0 as the root of all connections
        current_index = 0; prev_index = 0; advance = True
        chain_indices = [current_index]; chain_directions = []; chain_connections_i = []; chain_directions_i = []
        
        # We keep building up and down the chain until we either found an infinite loop (BAD_void) or we finished up all chain paths from index 0.
        while True:
            
            if advance:
                chain_connections_i.append([_ for _ in all_connections[current_index]]); chain_connections_i[-1][prev_index] = False
                chain_directions_i.append( [_ for _ in all_directions[ current_index]])
            
            # If we (still) have some connections (left) from this chain link (that are not this respective back and forth), we find the first one...
            if np.sum(chain_connections_i[-1]) != 0:
                new_index = np.argmax(chain_connections_i[-1])
                
                chain_connections_i[-1][new_index] = False
                direction = chain_directions_i[-1][new_index]
        
                chain_indices.append(new_index); chain_directions.append(direction)
                
                # If this new index already exists in the chain, we have formed a loop (0->1->2->0).
                if new_index in chain_indices[:-1]:
                    # We check if the loop is infinite or not by summing up all the directions from the first occurance of this patch in the chain up to present
                    #    and seeing if each direction component sums up to zero.
                    BAD_void = not np.array_equal(np.sum(chain_directions[np.argmax(np.array(chain_indices[:-1]) == new_index):], axis=0), np.array([0,0,0]))
                    if BAD_void: break
        
                    # If it isn't a BAD_void, we remove its last occurance from the chain and the directions.
                    chain_indices = chain_indices[:-1]; chain_directions = chain_directions[:-1]
                    advance = False
        
                # But if it isn't already in the chain, we look for its connections.
                else:
                    prev_index = current_index; current_index = new_index; advance = True
            
            # If we ran out of all connections for the current_index...
            else:
                # If this was the index 0 link, we are done!
                if current_index == 0: break
                    
                # Otherwise, we remove this chain link (and its direction) and return to the previous link.
                chain_indices = chain_indices[:-1]; chain_directions = chain_directions[:-1]
                current_index = chain_indices[-1]; advance = False
                chain_connections_i = chain_connections_i[:-1]; chain_directions_i = chain_directions_i[:-1]

        if BAD_void: BAD_voids[i1] = True; continue



        
    
    ### Combine the voids and give them an origin
    ################################################################
    
    
    patches = [[] for _ in range(no_patches)]
    for i,j,k in voids_edge_coords:
        patches[labeled_array[i][j][k]].append([i,j,k])
    patches = [np.array(_) for _ in patches]

    patches1 = [[] for _ in range(no_patches)]
    for i,j,k in np.argwhere(fg_100 == i1):
        patches1[labeled_array1[i][j][k]].append([i,j,k])
    patches1 = [np.array(_) for _ in patches1]
    
    
    chain_indices = [0]; chain_indices_not_checked = [0]
    chain_shifts  = [np.array([0,0,0]) for _ in range(no_patches)]
    
    while len(chain_indices) != no_patches:
        current_index = chain_indices_not_checked[0]; chain_indices_not_checked.remove(current_index)
        connections = all_connections[current_index]
        directions  = all_directions[ current_index]
        chain_shift = chain_shifts[   current_index]
    
        for index in range(no_patches):
            if connections[index] and (index not in chain_indices):
                chain_indices.append(index); chain_indices_not_checked.append(index)
                chain_shifts[index]  = chain_shift + size*directions[index]
                patches[ index]     += chain_shift + size*directions[index]
                patches1[index]     += chain_shift + size*directions[index]

    



    ### Remove the cells that do not neighbour any walls
    ####################################################

    patches_combined  = np.array([pc for pc in [_0 for _1 in patches  for _0 in _1] if check_neighbor_walls_cube(fg_100, pc, size)])
    patches_combined1 = np.array([pc for pc in [_0 for _1 in patches1 for _0 in _1]                                               ])
    
    with open(file_path_ellips_Z+"patches_combined/"+str(i1)+".pk", 'wb') as f: pkl.dump(patches_combined,  f)
    with open(file_path_ellips_Z+"void_combined/"   +str(i1)+".pk", 'wb') as f: pkl.dump(patches_combined1, f)

    


    

    ### Ellipsoid fitting
    #####################

    try:
        center, evecs, radii, v = ellipsoid_fit(patches_combined)
        BAD_fit = False
        for r in radii:
            # We set a max parameter as 3*size... it is arbitrary, sure, but after that point, even if we had a well constructed void that is
            #    longed than that and the ellipsoid_fit on it is good, we just exclude it.
            if (r >= 3*size) or (r <= 1): BAD_fit = True
        if not BAD_fit: l_center.append(center); l_radii.append(radii); l_evecs.append(evecs); l_v.append(v); l_i1.append(i1)
    
    except: continue
    

with open(file_path_ellips_Z+"BAD_voids.pk",                    'wb') as f: pkl.dump(BAD_voids,              f)
with open(file_path_ellips_Z+"voids_multiple_patches.pk",       'wb') as f: pkl.dump(voids_multiple_patches, f)
with open(file_path_ellips_Z+"voids_edge_coords/voids_used.pk", 'wb') as f: pkl.dump(voids_used, f)

with open(file_path_ellips_Z+"Ellipsoid_values/l_center.pk",    'wb') as f: pkl.dump(l_center,               f)
with open(file_path_ellips_Z+"Ellipsoid_values/l_radii.pk",     'wb') as f: pkl.dump(l_radii,                f)
with open(file_path_ellips_Z+"Ellipsoid_values/l_evecs.pk",     'wb') as f: pkl.dump(l_evecs,                f)
with open(file_path_ellips_Z+"Ellipsoid_values/l_v.pk",         'wb') as f: pkl.dump(l_v,                    f)
with open(file_path_ellips_Z+"Ellipsoid_values/l_i1.pk",        'wb') as f: pkl.dump(l_i1,                   f)

---
---
---